In [ ]:
import io
import pandas as pd

# 1. Criando um CSV simulado em memória
# Note que ele usa ';' como separador, possui 2 linhas de metadados antes da tabela
# e usa 'N/A' e 'MISSING' para indicar dados ausentes.

csv_dados = """
-- Relatório Semanal de Vendas
-- Gerado automaticamente em 2026-07-27
id_venda;cliente_cpf;data_venda;valor_total;status
101;01234567890;2026-07-01;150.50;Concluido
102;09876543210;2026-07-02;N/A;Pendente
103;11122233344;2026-07-03;89.90;MISSING
104;55566677788;2026-07-04;1200.00;Concluido
105;99988877766;2026-07-05;45.00;Cancelado
"""

# 2. Leitura com aplicação dos parâmetros
df = pd.read_csv(
    # filepath_or_buffer: usando o buffer do StringIO
    filepath_or_buffer=io.StringIO(csv_dados),
    # sep: informando que o delimitador é ponto e vírgula
    sep=";",
    # skiprows: pulando as 2 primeiras linhas de metadados
    skiprows=2,
    # header: definindo que a linha 0 (após o skip) contém os nomes das colunas
    header=0,
    # index_col: usando a coluna 'id_venda' como o índice do DataFrame
    index_col="id_venda",
    # usecols: selecionando apenas as colunas necessárias
    usecols=["id_venda", "cliente_cpf", "valor_total", "status"],
    # dtype: garantindo que o CPF permaneça como string (evita perder o '0' à esquerda)
    dtype={"cliente_cpf": str},
    # na_values: tratando os valores customizados de dados nulos
    na_values=["N/A", "MISSING"],
    # nrows: lendo apenas as primeiras 4 linhas de dados válidos
    nrows=4,
    # encoding: especificando o padrão de caracteres
    encoding="utf-8",
)

print(df)
print("\nTipos das colunas (dtypes):")
print(df.dtypes)

          cliente_cpf  valor_total     status
id_venda                                     
101       01234567890        150.5  Concluido
102       09876543210          NaN   Pendente
103       11122233344         89.9        NaN
104       55566677788       1200.0  Concluido

Tipos das colunas (dtypes):
cliente_cpf     object
valor_total    float64
status          object
dtype: object


In [ ]:
import io
import pandas as pd

# 1. HTML simulado contendo hiperlinks, classes, id, valores numéricos e linhas extras
html_conteudo = """
<html>
  <body>
    <!-- Tabela 1: Lixo / Não desejada -->
    <table class="outra-tabela">
      <tr><td>Apenas dados irrelevantes</td></tr>
    </table>

    <!-- Tabela 2: Tabela Principal de Vendas -->
    <table id="relatorio-vendas" class="tabela-dados">
      <thead>
        <tr><th colspan="4">RELATÓRIO DE VENDAS REGIONAIS 2026</th></tr>
        <tr><th>Código</th><th>Cidade</th><th>População</th><th>Faturamento</th></tr>
      </thead>
      <tbody>
        <tr>
          <td>101</td>
          <td><a href="https://pt.wikipedia.org/wiki/Teresina">Teresina</a></td>
          <td>868.523</td>
          <td>R$ 4.500,50</td>
        </tr>
        <tr>
          <td>102</td>
          <td><a href="https://pt.wikipedia.org/wiki/Parnaiba">Parnaíba</a></td>
          <td>153.078</td>
          <td>R$ 1.200,75</td>
        </tr>
        <tr>
          <td>103</td>
          <td><a href="https://pt.wikipedia.org/wiki/Picos">Picos</a></td>
          <td>78.000</td>
          <td>R$ 850,20</td>
        </tr>
      </tbody>
    </table>
  </body>
</html>
"""

# 2. Primeira leitura: números limpos
# thousands/decimal funcionam aqui porque as células ainda são strings simples
tabela_numeros = pd.read_html(
    io=io.StringIO(html_conteudo),
    # match: Filtra tabelas que contenham a palavra 'Teresina'
    match="Teresina",
    # match: Filtra tabelas que contenham a palavra 'Teresina'
    flavor="bs4",
    # attrs: Filtra a tabela especificamente pelo atributo id do HTML
    attrs={"id": "relatorio-vendas"},
    # skiprows: Pula a 1ª linha do thead ("RELATÓRIO DE VENDAS REGIONAIS 2026")
    skiprows=1,
    # header: Define a linha seguinte (agora linha 0) como o cabeçalho real das colunas
    header=0,
    # index_col: Usa a coluna 'Código' (posição 0) como índice do DataFrame
    index_col=0,
    # thousands / decimal: Define os separadores para parsing numérico
    thousands=".",
    decimal=",",
)[0]

# 3. Segunda leitura: só pra pegar os links
# Aqui cada célula vira uma tupla (texto, link) — inclusive o índice
tabela_links = pd.read_html(
    io=io.StringIO(html_conteudo),
    match="Teresina",
    flavor="bs4",
    attrs={"id": "relatorio-vendas"},
    skiprows=1,
    header=0,
    index_col=0,
    # extract_links: Extrai as tuplas (texto, link) da tag <a> na coluna Cidade
    extract_links="body",  # "body" evita bagunçar o cabeçalho; "all" bagunçaria tudo
)[0]

# 4. Junta os dois: pega só o link (segundo item da tupla) da coluna Cidade
# Usamos .values porque o índice de tabela_links também virou tupla, então
# não dá pra juntar por índice — usamos a ordem das linhas mesmo.
df = tabela_numeros.copy()
df["Link"] = tabela_links["Cidade"].apply(lambda t: t[1]).values

print(df)
print()
print(df.dtypes)
print()
print("Link da Teresina:", df.loc[101, "Link"])

          Cidade  População  Faturamento  \
Código                                     
101     Teresina     868523  R$ 4.500,50   
102     Parnaíba     153078  R$ 1.200,75   
103        Picos      78000    R$ 850,20   

                                          Link  
Código                                          
101     https://pt.wikipedia.org/wiki/Teresina  
102     https://pt.wikipedia.org/wiki/Parnaiba  
103        https://pt.wikipedia.org/wiki/Picos  

Cidade         object
População       int64
Faturamento    object
Link           object
dtype: object

Link da Teresina: https://pt.wikipedia.org/wiki/Teresina


In [ ]:
import pandas as pd

# 1. URL da página da Wikipedia
url = "https://pt.wikipedia.org/wiki/Lista_de_munic%C3%ADpios_do_Piau%C3%AD_por_popula%C3%A7%C3%A3o"

# 2. Leitura usando o parâmetro 'match' para ir direto na tabela certa
# Usamos 'match' com uma palavra que sabemos que está no cabeçalho ou corpo da tabela
tabelas = pd.read_html(url, match="Censo")

# Como read_html retorna uma lista, pegamos o primeiro item [0]
df_municipios = tabelas[0]

# Exibindo os primeiros resultados
print(df_municipios.head())

HTTPError: HTTP Error 403: Forbidden

In [ ]:
import os
import requests
import pandas as pd

url = "https://pt.wikipedia.org/wiki/Lista_de_munic%C3%ADpios_do_Piau%C3%AD_por_popula%C3%A7%C3%A3o"

# Simulando um navegador real no User-Agent
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

# Faz a requisição HTTP primeiro
resposta = requests.get(url, headers=headers)

# Passa o conteúdo HTML textual (resposta.text) para o pandas
df = pd.read_html(io.StringIO(resposta.text), match="Censo")[0]

df

,Pos.,Município,Estimativa 2025,Censo 2022
0,–,Piauí,3.384.547,3.271.199
1,Acima de 50.000 habitantes,Acima de 50.000 habitantes,Acima de 50.000 habitantes,Acima de 50.000 habitantes
2,1,Teresina,905.692,866.300
3,2,Parnaíba,170.491,162.159
4,3,Picos,86.701,83.090
...,...,...,...,...
225,219,Tanque do Piauí,2.319,2.316
226,221,São Miguel da Baixa Grande,2.299,2.269
227,220,São Luis do Piauí,2.277,2.309
228,223,Santo Antônio dos Milagres,2.169,2.138


In [ ]:
import pandas as pd
import sqlite3

# 1. Criando um banco de dados SQLite temporário em memória RAM
conexao = sqlite3.connect(":memory:")

# Criando e populando uma tabela para o teste
conexao.execute(
    """
    CREATE TABLE vendas (
        id INTEGER PRIMARY KEY,
        cliente TEXT,
        data_venda TEXT,
        valor REAL
    )
"""
)
conexao.execute(
    """
    INSERT INTO vendas VALUES
    (1, 'Ana', '2026-07-01', 250.00),
    (2, 'Bruno', '2026-07-02', 120.50),
    (3, 'Carla', '2026-07-03', 890.00)
"""
)

# 2. Executando o pd.read_sql com seus principais parâmetros
query_sql = "SELECT id, cliente, data_venda, valor FROM vendas WHERE valor > 100"

df = pd.read_sql(
    # sql: A instrução SQL que será executada no banco
    sql=query_sql,
    # con: O objeto de conexão com o banco de dados
    con=conexao,
    # index_col: Define qual coluna do banco será usada como índice do DataFrame
    index_col="id",
    # parse_dates: Converte automaticamente a coluna de texto do banco para datetime
    parse_dates=["data_venda"],
    # columns: (opcional) usado apenas se você passar o nome de uma tabela em 'sql' em vez da query
    columns=None,
)

print(df)
print("\nTipos de dados importados:")
print(df.dtypes)

   cliente data_venda  valor
id                          
1      Ana 2026-07-01  250.0
2    Bruno 2026-07-02  120.5
3    Carla 2026-07-03  890.0

Tipos de dados importados:
cliente               object
data_venda    datetime64[ns]
valor                float64
dtype: object


In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# O SQLAlchemy faz a ponte de conexão com o banco
engine = create_engine("SUA_URL_DO_BANCO")

query = "SELECT * FROM vendas_loja;"

# O Pandas usa essa ponte para ler a tabela e transformar em DataFrame
df = pd.read_sql(query, engine)

In [ ]:
import pandas as pd

df =  pd.read_csv('BO_2007_1.csv', low_memory=False)
df['IDADE_PESSOA'].unique()



array(['23', nan, '34', '42', '26', '32', '24', '35', '3', '27', '31',
       '41', '25', '21', '30', '20', '39', '50', '40', '33', '72', '6',
       '44', '18', '22', '16', '43', '37', '19', '36', '46', '29', '45',
       '15', '28', '38', '12', '48', '8', '49', '54', '56', '13', '75',
       '66', '52', '68', '69', '55', '51', '107', '7', '17', '64', '47',
       '92', '9', '59', '58', '63', '73', '53', '5', '14', '60', '74',
       '77', '80', '62', '71', '57', '0', '81', '70', '11', '2', '65',
       '79', '78', '61', '76', '86', '4', '67', '150', '83', '10', '82',
       '1', '87', '84', '85', '91', '88', '89', '95', '93', '90',
       '22 ANOS                 ', '94', '118', '104', '99', '103', '119',
       '130', '169', '97'], dtype=object)

In [ ]:
# teste_idade_int = pd.to_numeric(df['IDADE_PESSOA'])
teste_idade_int_2 = df["IDADE_PESSOA"].astype(int)
teste_idade_int_2


ValueError: cannot convert float NaN to integer

In [ ]:
# 1. Limpar strings da coluna IDADE_PESSOA
# Substituir ' ANOS' e espaços em branco por uma string vazia
df['IDADE_PESSOA'] = df['IDADE_PESSOA'].fillna(0)

df['IDADE_PESSOA'] = df['IDADE_PESSOA'].astype(str).str.replace(' ANOS', '', regex=False).str.strip()

# 2. Converter a coluna para numérico, forçando erros para NaN
df['IDADE_PESSOA'] = pd.to_numeric(df['IDADE_PESSOA'])

# 3. Tratar valores ausentes (NaN) preenchendo com 0 (ou outra estratégia, como mediana/média)

# 4. Converter a coluna para inteiro


print("✅ Coluna 'IDADE_PESSOA' limpa e convertida para tipo inteiro.")
print("Valores únicos após a limpeza e conversão:")
print(df['IDADE_PESSOA'].unique())
print("Tipo de dados da coluna 'IDADE_PESSOA' agora é:", df['IDADE_PESSOA'].dtype)

✅ Coluna 'IDADE_PESSOA' limpa e convertida para tipo inteiro.
Valores únicos após a limpeza e conversão:
[ 23   0  34  42  26  32  24  35   3  27  31  41  25  21  30  20  39  50
  40  33  72   6  44  18  22  16  43  37  19  36  46  29  45  15  28  38
  12  48   8  49  54  56  13  75  66  52  68  69  55  51 107   7  17  64
  47  92   9  59  58  63  73  53   5  14  60  74  77  80  62  71  57  81
  70  11   2  65  79  78  61  76  86   4  67 150  83  10  82   1  87  84
  85  91  88  89  95  93  90  94 118 104  99 103 119 130 169  97]
Tipo de dados da coluna 'IDADE_PESSOA' agora é: int64
